# Step 4 · Modeling (모델 훈련 — Google Colab)

> **목표:** 데이터셋으로 CNN 모델을 GPU로 훈련합니다.
> ⚠️ **이 노트북은 Google Colab에서 실행합니다.** (무료 GPU 사용)

**오늘 배우는 것**
- 🧠 CNN이 이미지에서 **조향 패턴을 어떻게 배우는지**
- ⚙️ `epochs`, `batch_size`, `learning_rate` 파라미터의 의미

## Part 0. 개념 이해

### 우리 모델: Transfer Learning (전이학습)

이 단계에서는 **Transfer Learning(전이학습)** 으로 CNN 모델을 훈련합니다.

| 단계 | 설명 |
|------|------|
| ① 기반 모델 | 구글의 **MobileNetV3** — ImageNet(100만 장+)으로 미리 훈련된 모델 |
| ② 가져오기 | 특징 추출 레이어를 **그대로 가져옵니다** (가중치 고정) |
| ③ 추가 훈련 | 우리 도로 데이터로 **마지막 레이어 3개만** 추가 훈련 |
| ④ 결과 | 데이터 수백~수천 장으로도 좋은 성능 |

**모델 구조:**

```
구글 MobileNetV3 (고정 — 변경 안 함)
  ↓
우리 레이어 3개 (훈련)
  ↓
조향값 출력 (-1.0 ~ +1.0)
```

### Transfer Learning vs 순수 CNN 비교

|  | Transfer Learning (우리) | 순수 CNN |
|--|--------------------------|----------|
| **필요 데이터** | 수백~수천 장 | 수만 장 이상 |
| **훈련 시간** | 빠름 (10~20분) | 느림 |
| **성능** | 좋음 | 데이터 많아야 좋음 |

### 핵심 도구

| 도구 | 역할 |
|------|------|
| **TensorFlow** | AI 훈련 엔진 (구글이 만든 오픈소스) |
| **Keras** | TensorFlow를 더 쉽게 쓰는 인터페이스 |

### 왜 Google Colab인가요?

| 환경 | 훈련 시간 |
|------|-----------|
| 일반 노트북 PC (CPU) | 수 시간 |
| Google Colab (T4 GPU) | **10~20분** |

Colab은 무료 GPU를 제공합니다.

## 🗺️ 이 단계 전체 그림

```
Step 4 노트북(.ipynb)
  → Colab에서 열기 (상단 Open in Colab 버튼)
  → T4 GPU 선택
  → 데이터셋(.zip) 업로드
  → Transfer Learning 훈련 (약 10~20분)
  → laneD1.keras 다운로드
  → PC의 models/laneD1/ 폴더에 복사
```

## Part 1. Colab으로 훈련 (권장)
> **무료 GPU — 10~20분 완료**

아래 버튼을 클릭하면 Google Colab에서 바로 열립니다.
**위에서 아래로 순서대로 실행**하세요. 복붙 불필요.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brainai-hub/brainai-car-colab/blob/main/Step%204%20-%20Modeling%20(Colab).ipynb)

### 라이브러리 설치 (Colab 전용)

설치 후 런타임이 자동 재시작됩니다 — 이건 정상입니다.

In [ ]:
# Colab 전용 — PC Jupyter에서는 실행하지 마세요
!pip install tensorflow==2.19.0 keras==3.11.2

import os
os.kill(os.getpid(), 9)   # 런타임 재시작 (정상)

In [ ]:
# 재시작 후 버전 확인
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")    # 2.19.0
print(f"Keras:      {tf.keras.__version__}")  # 3.11.2
print(f"GPU:        {len(tf.config.list_physical_devices('GPU')) > 0}")  # True여야 함

### 모델 코드 정의 (Colab · PC 공통)

아래 셀을 실행하면 `TrainingConfig` / `CNNTrainer` 클래스가 메모리에 로드됩니다.
**Colab · PC 양쪽에서 모두 실행해야 합니다.**

In [ ]:
# ── Transfer Learning 모델 클래스 정의 ────────────────────────────────────────
# Colab · PC 공통 — 이 셀을 실행하면 TrainingConfig / CNNTrainer 가 메모리에 로드됩니다.

import json
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models


@dataclass
class TrainingConfig:
    """훈련 설정값을 담는 데이터클래스."""
    dataset_path: str
    model_name: str = "model"
    epochs: int = 30
    batch_size: int = 32
    learning_rate: float = 0.001
    image_size: tuple = (224, 224)
    freeze_base: bool = True
    output_dir: str = "models"

    def print_summary(self):
        print("=" * 50)
        print(f"  dataset_path  : {self.dataset_path}")
        print(f"  model_name    : {self.model_name}")
        print(f"  epochs        : {self.epochs}")
        print(f"  batch_size    : {self.batch_size}")
        print(f"  learning_rate : {self.learning_rate}")
        print(f"  image_size    : {self.image_size}")
        print(f"  freeze_base   : {self.freeze_base}")
        print(f"  output_dir    : {self.output_dir}")
        print("=" * 50)


class DataLoader:
    """이미지 경로와 조향값(steering)을 로드합니다."""

    def __init__(self, dataset_path: str, image_size: tuple = (224, 224)):
        self.dataset_path = dataset_path
        self.image_size = image_size

    def load_split(self, split: str) -> tuple[np.ndarray, np.ndarray]:
        images_dir = os.path.join(self.dataset_path, split, "images")
        annotations_dir = os.path.join(self.dataset_path, split, "annotations")
        if not os.path.exists(images_dir):
            raise FileNotFoundError(f"Not found: {images_dir}")
        img_paths = sorted(Path(images_dir).glob("*.jpg"))
        X, y = [], []
        for img_path in img_paths:
            json_path = Path(annotations_dir) / (img_path.stem + ".json")
            if not json_path.exists():
                continue
            try:
                with open(json_path) as f:
                    steering = float(json.load(f)["steering"])
            except (KeyError, ValueError, json.JSONDecodeError):
                continue
            img = tf.keras.utils.load_img(str(img_path), target_size=self.image_size)
            X.append(tf.keras.utils.img_to_array(img) / 255.0)
            y.append(steering)
        print(f"  [{split}] {len(X)} samples loaded")
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


class ImageAugmentation:
    """훈련 데이터 증강 (좌우 반전 · 밝기 조정 · 가우시안 노이즈)."""

    def __init__(self, flip_prob: float = 0.5, brightness_range: float = 0.2,
                 noise_std: float = 0.02):
        self.flip_prob = flip_prob
        self.brightness_range = brightness_range
        self.noise_std = noise_std

    def augment(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        X_aug, y_aug = list(X), list(y)
        for img, steering in zip(X, y):
            if random.random() < self.flip_prob:
                X_aug.append(np.fliplr(img))
                y_aug.append(-steering)
            delta = random.uniform(-self.brightness_range, self.brightness_range)
            X_aug.append(np.clip(img + delta, 0.0, 1.0))
            y_aug.append(steering)
            noisy = np.clip(img + np.random.normal(0, self.noise_std, img.shape), 0.0, 1.0)
            X_aug.append(noisy.astype(np.float32))
            y_aug.append(steering)
        idx = list(range(len(X_aug)))
        random.shuffle(idx)
        return (np.array(X_aug, dtype=np.float32)[idx],
                np.array(y_aug, dtype=np.float32)[idx])


class CNNTrainer:
    """
    Transfer Learning 기반 조향값 회귀 모델 훈련기.
    MobileNetV3Large (고정) → Dense(64) → Dropout(0.2) → Dense(32) → Dense(1)
    """

    def __init__(self, config: TrainingConfig):
        self.config = config
        self.model = self._build_model()

    def _build_model(self) -> keras.Model:
        base = keras.applications.MobileNetV3Large(
            input_shape=(*self.config.image_size, 3),
            include_top=False,
            weights="imagenet",
            minimalistic=False,
            alpha=0.75,
        )
        base.trainable = not self.config.freeze_base
        model = models.Sequential([
            base,
            layers.GlobalAveragePooling2D(),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.2),
            layers.Dense(32, activation="relu"),
            layers.Dense(1),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.config.learning_rate),
            loss="mse",
            metrics=["mae"],
        )
        return model

    def train(self) -> tuple[keras.Model, keras.callbacks.History]:
        loader = DataLoader(self.config.dataset_path, self.config.image_size)
        print("\n[1/3] 데이터 로딩...")
        X_train, y_train = loader.load_split("train")
        X_val,   y_val   = loader.load_split("validation")
        print("\n[2/3] 데이터 증강...")
        aug = ImageAugmentation()
        X_train, y_train = aug.augment(X_train, y_train)
        print(f"  증강 후 훈련 샘플: {len(X_train)}")
        print("\n[3/3] 훈련 시작...")
        os.makedirs(os.path.join(self.config.output_dir, self.config.model_name), exist_ok=True)
        ckpt_path = os.path.join(
            self.config.output_dir, self.config.model_name,
            f"{self.config.model_name}.keras"
        )
        callbacks = [
            keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, verbose=1),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        ]
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )
        print(f"\n✅ 훈련 완료! 모델 저장: {ckpt_path}")
        return self.model, history


print("✅ 모델 클래스 로드 완료 (TrainingConfig, CNNTrainer)")

### 데이터셋 업로드 및 압축 해제

Colab 왼쪽 📁 아이콘 → `laneD1_Dataset.zip` 업로드 후 아래 셀 실행

In [ ]:
!mkdir -p /content/datasets
!unzip -q laneD1_Dataset.zip -d /content/datasets/

import os
t = len(os.listdir("/content/datasets/laneD1_Dataset/train/images"))
v = len(os.listdir("/content/datasets/laneD1_Dataset/validation/images"))
print(f"훈련: {t}장  |  검증: {v}장")

### 설정 및 훈련 실행

`epochs` · `batch_size` · `learning_rate` 를 조정한 뒤 셀을 실행하면 훈련이 시작됩니다.

> ⏱️ 예상 시간: 약 10~30분

In [ ]:
# ⬇️ 여기만 바꾸세요
config = TrainingConfig(
    dataset_path="/content/datasets/laneD1_Dataset",
    model_name="laneD1",
    epochs=30,       # 전체 데이터를 몇 번 반복 학습할지
    batch_size=32,   # 한 번에 학습할 이미지 수
    learning_rate=0.001,
)
config.print_summary()

trainer = CNNTrainer(config)
model, history = trainer.train()

### 모델 다운로드

> ⚠️ Colab은 세션이 끊기면 파일이 사라집니다. 훈련 직후 반드시 다운로드!

In [ ]:
from google.colab import files
import os

model_path = "models/laneD1/laneD1.keras"
if os.path.exists(model_path):
    files.download(model_path)
    print(f"✅ 다운로드 시작: {model_path}")
    print("다음: 다운로드된 laneD1.keras를 PC의 models/laneD1/ 폴더에 넣으세요")
else:
    print("❌ 모델 파일이 없습니다. 훈련이 완료됐는지 확인하세요.")

## Part 2. PC에서 훈련 (선택)
> **GPU 없이 로컬 실행 — 수 시간 소요**

> ⚠️ **주의:** GPU 없으면 에포크당 수 분 → 전체 30 에포크 수 시간. **Colab 강력 권장.**

Jupyter Lab에서 이 노트북을 열고 다음 순서로 실행하세요.
1. Part 1의 **모델 코드 정의 셀** 실행 (`TrainingConfig` / `CNNTrainer` 로드)
2. 아래 설정 및 훈련 셀 실행 (pip install · 데이터셋 업로드 셀은 건너뜀)

### 설정 및 훈련 실행

`dataset_path` 를 Step 3에서 만든 로컬 경로로 바꾼 뒤 실행하세요.

> ⏱️ CPU만 있으면 에포크당 수 분 → 전체 수 시간 소요. Colab 강력 권장.

In [ ]:
# ⬇️ 여기만 바꾸세요 (로컬 실행용)
config = TrainingConfig(
    dataset_path="dataset/laneD1_Dataset",  # Step 3에서 생성한 로컬 경로
    model_name="laneD1",
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
)
config.print_summary()

trainer = CNNTrainer(config)
model, history = trainer.train()

## 다음 ▶️ Step 5 · Model Optimization

`laneD1.keras` 파일을 PC `models/laneD1/` 폴더에 복사한 뒤 Step 5를 실행하세요.